**[Course Home](../README.md) | [Python Lab](../python/08_priority_queues_and_scheduling_disciplines.ipynb) | [Prev: Ch 07](07_queues_with_breakdowns_and_vacations.ipynb) | [Next: Ch 09](09_jackson_networks_and_bottleneck_routing.ipynb)**

*Part III: Scheduling, Networks & Global Fleet Optimization — Chapter 08 of 11*

---

# Chapter 8: Scheduling & Preemption — Priority Queues, The $c\mu$-Rule & Tail Latency Control
### *FIFO Breakdown, Head-of-Line Blocking, SRPT Optimality, Cobham's Multi-Class Laws, and Cost-Optimal Prioritization*

---

## 1. Why First-In First-Out (FIFO) Fails Under Variance

In basic operating systems and networking textbooks, First-In First-Out (FIFO / FCFS) is treated as the default, "fair" queue discipline.
In production computing fleets—continuous integration test farms, database query engines, and cloud API gateways—FIFO is often the **worst possible** scheduling policy:

1. **Head-of-Line (HoL) Blocking**:
   When job sizes exhibit high variance (e.g. standard deviation exceeds the mean, $c_s > 1$), an enormous job ("an elephant", such as a 2-hour end-to-end integration test suite) blocks all subsequent jobs in the queue. Dozens of lightweight jobs ("mice", such as 15-second lint or unit tests) sit idle waiting for the elephant to vacate the server.
2. **Indifference to Delay Economics**:
   Under FIFO, an urgent interactive hotfix pull request submitted by an engineer waiting in real time is treated with the exact same priority as a low-urgency batch documentation rebuild.
3. **Sub-Optimal System-Wide Mean Delay**:
   As proven by Linus Schrage (1968), FIFO does not minimize average waiting time or queue length. Scheduling shorter jobs before longer jobs dramatically reduces the aggregate number of waiting jobs in the system.

This chapter develops the mathematical principles of scheduling and preemption, deriving the exact multi-class formulas of Alan Cobham, proving the economic optimality of the $c\mu$-rule, and providing concrete mechanisms for tail latency control in production fleets.

---

## 2. The Landscape of Scheduling Disciplines

Queue scheduling disciplines govern the order in which waiting jobs are selected for service by available servers:

| Discipline | Description | Preemptive? | Requires Job Size Knowledge? | Minimizes Mean Latency? |
| :--- | :--- | :---: | :---: | :---: |
| **FIFO** | First-In First-Out (FCFS) | No | No | No (Poor under variance) |
| **LIFO** | Last-In First-Out | No / Yes | No | No (High variance) |
| **PS** | Processor Sharing (Fair Queuing) | Continuous | No | Robust against variance |
| **SJF** | Shortest Job First | No | Yes (Known duration) | Optimal among non-preemptive non-priority |
| **SRPT** | Shortest Remaining Processing Time | Yes | Yes (Known remaining duration) | **Strictly Optimal across ALL causal policies** |
| **Priority** | Multi-class strict priority | Non / Preemptive | Class label only | Optimal under $c\mu$-rule |

### Schrage's Optimality Theorem (1968)

**Theorem**: Among all causal, work-conserving scheduling policies on a single server, **Shortest Remaining Processing Time (SRPT)** strictly minimizes:
1. The total number of jobs in the system at all times $N(t)$,
2. The expected queue length $\mathbb{E}[N_q]$,
3. The expected mean sojourn time $\mathbb{E}[T] = \mathbb{E}[W_q] + \mathbb{E}[S]$.

**Intuitive Proof**:
At any instant $t$, to minimize the number of uncompleted jobs, the server should maximize the departure rate of jobs in the immediate future. The job closest to completion requires the least remaining server time $\Delta t$ to decrement the active job count $N(t)$ by 1. Therefore, spending server effort on any job other than the one with the shortest remaining processing time strictly delays the next departure without accelerating any other, leaving $N(t)$ strictly larger.

---

## 3. Multi-Class Priority Queues: Cobham's Conservation Law

When jobs can be classified into priority tiers (e.g. High Priority Tier 1 vs. Low Priority Tier 2), we can establish strict priority disciplines without needing continuous execution tracking.

Consider a single server fed by $P$ independent Poisson arrival streams with rates $\lambda_1, \lambda_2, \dots, \lambda_P$.
Each class $p \in \{1, \dots, P\}$ has service time distribution $S_p$ with mean $\mathbb{E}[S_p] = 1/\mu_p$ and second moment $\mathbb{E}[S_p^2]$.
Class $1$ has the highest priority; Class $P$ has the lowest.
The utilization of class $p$ is $\rho_p = \lambda_p \mathbb{E}[S_p]$, and the total utilization is:

$$
\rho = \sum_{p=1}^P \rho_p < 1
$$

Let the cumulative utilization of the top $p$ priority classes be denoted by:

$$
\sigma_p = \sum_{i=1}^p \rho_i, \quad \text{with } \sigma_0 = 0
$$

### Non-Preemptive Priority (Cobham's Theorem, 1954)

In non-preemptive priority, a high-priority arrival cannot interrupt an ongoing job. If a server is busy with a Class $P$ job when a Class 1 job arrives, the Class 1 job must wait until the current job finishes, but jumps ahead of all waiting Class 2..P jobs.

By renewal theory, the mean residual service time of the job currently occupying the server (if busy) is:

$$
W_0 = \sum_{i=1}^P \frac{\lambda_i \mathbb{E}[S_i^2]}{2}
$$

Alan Cobham proved that the expected waiting time in queue for a Class $p$ job is given by:

$$
\mathbb{E}[W_{q, p}] = \frac{W_0}{(1 - \sigma_{p-1})(1 - \sigma_p)} = \frac{W_0}{\left(1 - \sum_{i=1}^{p-1} \rho_i\right)\left(1 - \sum_{i=1}^p \rho_i\right)}
$$

### Examining the Formula for Class 1 (Highest Priority)
For Class 1 ($p=1$):

$$
\mathbb{E}[W_{q, 1}] = \frac{W_0}{1 - \rho_1}
$$

Class 1 jobs only wait for:
1. The residual service time $W_0$ of whatever job was already in service upon arrival,
2. Other Class 1 jobs that arrived before them.
Class 1 is **completely shielded** from the queue congestion of lower classes ($\rho_2, \dots, \rho_P$)!

### Examining the Formula for Low-Priority Classes
For the lowest class $P$:

$$
\mathbb{E}[W_{q, P}] = \frac{W_0}{(1 - \sigma_{P-1})(1 - \rho)}
$$

As total utilization $\rho \to 1^-$, the denominator $(1 - \rho)$ explodes to zero. The lowest priority class absorbs the entire heavy-traffic congestion of the system!

### Preemptive-Resume Priority

If high-priority jobs immediately preempt lower-priority jobs (interrupting execution and resuming later where they left off):
For Class 1:

$$
\mathbb{E}[W_{q, 1}^{\text{preempt}}] = \frac{\frac{\lambda_1 \mathbb{E}[S_1^2]}{2}}{1 - \rho_1}
$$

Class 1 behaves as if lower-priority classes do not exist at all! Even the residual service time $W_0$ of lower classes is eliminated from their waiting time.

---

## 4. The $c\mu$-Rule: Delay Cost Minimization

In production systems, not all delays carry the same cost.
Suppose each job of class $i$ incurs a delay cost of $C_i$ dollars per unit time spent waiting in the system (or developer delay penalty weight).
Our objective is to find the scheduling policy that minimizes the long-term aggregate delay cost rate:

$$
\min \sum_{i=1}^P C_i \mathbb{E}[N_{q, i}] = \min \sum_{i=1}^P C_i \lambda_i \mathbb{E}[W_{q, i}]
$$

### The Optimality Theorem (W.E. Smith 1956, Kakalik 1965)

**Theorem ($c\mu$-Rule)**: To minimize the total expected holding and delay cost, non-preemptive priority should be assigned to classes in descending order of the product:

$$
C_i \cdot \mu_i = \frac{C_i}{\mathbb{E}[S_i]}
$$

That is, Class $j$ should have strictly higher priority than Class $k$ if and only if:

$$
C_j \mu_j > C_k \mu_k \iff \frac{C_j}{\mathbb{E}[S_j]} > \frac{C_k}{\mathbb{E}[S_k]}
$$

### Economic Intuition: "Cost Reduction per Server Second"
The index $\frac{C_i}{\mathbb{E}[S_i]}$ represents the **marginal cost reduction per second of server time consumed**:
- $C_i$: Delay cost burned per second while this job waits.
- $\mathbb{E}[S_i]$: Number of server seconds required to clear this job out of the system.
- Ratio: By serving the job with highest $C_i / \mathbb{E}[S_i]$, the server relieves the greatest amount of cost penalty per second of effort!

### Special Cases of the $c\mu$-Rule:
1. **Equal Costs ($C_1 = C_2 = \dots = C_P$)**:
   The $c\mu$-rule reduces to prioritizing by $\mu_i$ (descending), which is equivalent to prioritizing by $\mathbb{E}[S_i]$ (ascending). This recovers **Shortest Job First (SJF)**!
2. **Equal Durations ($\mathbb{E}[S_1] = \mathbb{E}[S_2] = \dots = \mathbb{E}[S_P]$)**:
   The $c\mu$-rule reduces to prioritizing by $C_i$ (descending). The server serves the most expensive jobs first!

---

## 5. Kleinrock's Conservation Law

Can an ingenious scheduling policy reduce the queue waiting time of *all* classes simultaneously without adding servers?

**Kleinrock's Conservation Law (1965)**:
For any non-preemptive, work-conserving queue discipline on a single server, the weighted sum of expected waiting times across all classes is **strictly invariant**:

$$
\sum_{i=1}^P \rho_i \mathbb{E}[W_{q, i}] = \frac{\rho W_0}{1 - \rho} = \text{Constant}
$$

**Engineering Reality**:
Priority scheduling is a **zero-sum game** with respect to total work. You cannot accelerate one class of jobs without penalizing another. What the $c\mu$-rule accomplishes is shifting the inevitable waiting delay into the classes that are cheapest ($C_i$ small) and longest ($\mathbb{E}[S_i]$ large).

---

## 6. Concrete Step-by-Step Worked Example with Real Numbers

Let us analyze a multi-class continuous integration test farm serving a software organization.

### The Three Workload Classes:
1. **Class 1 (Interactive PR Smoke Tests)**:
   - Fast smoke tests run by developers waiting actively at their workstations.
   - Arrival rate: $\lambda_1 = 30\text{ jobs/hour}$.
   - Mean duration: $\mathbb{E}[S_1] = 2\text{ minutes} = \frac{1}{30}\text{ hour} \implies \mu_1 = 30\text{ jobs/hour}$.
   - Utilization: $\rho_1 = \lambda_1 \mathbb{E}[S_1] = 30 \times \frac{1}{30} = 1.00$ (across a multi-server fleet; for single-server scale, let us normalize).

Let us calibrate to a single normalized server pipeline with total $\rho = 0.85$:
- **Class 1 (Smoke Tests)**:
  - $\lambda_1 = 12\text{ jobs/hour}$.
  - $\mathbb{E}[S_1] = 1.0\text{ minute} = \frac{1}{60}\text{ hr} \implies \mu_1 = 60\text{ jobs/hr}$.
  - $\sigma_{S1} = 0.5\text{ min} \implies \mathbb{E}[S_1^2] = 1.0^2 + 0.5^2 = 1.25\text{ min}^2 = 0.000347\text{ hr}^2$.
  - Utilization: $\rho_1 = 12 \times \frac{1}{60} = 0.20$.
  - Delay Cost: $C_1 = \$300/\text{hr}$ (active developer waiting in real time).
- **Class 2 (Standard PR Pre-Merge Tests)**:
  - $\lambda_2 = 3\text{ jobs/hour}$.
  - $\mathbb{E}[S_2] = 8.0\text{ minutes} = \frac{8}{60}\text{ hr} \implies \mu_2 = 7.5\text{ jobs/hr}$.
  - $\sigma_{S2} = 4.0\text{ min} \implies \mathbb{E}[S_2^2] = 8.0^2 + 4.0^2 = 80.0\text{ min}^2 = 0.02222\text{ hr}^2$.
  - Utilization: $\rho_2 = 3 \times \frac{8}{60} = 0.40$.
  - Delay Cost: $C_2 = \$60/\text{hr}$ (background PR merge queue).
- **Class 3 (Nightly Full Regression & Performance)**:
  - $\lambda_3 = 0.5\text{ jobs/hour}$.
  - $\mathbb{E}[S_3] = 30.0\text{ minutes} = \frac{30}{60} = 0.5\text{ hr} \implies \mu_3 = 2.0\text{ jobs/hr}$.
  - $\sigma_{S3} = 15.0\text{ min} \implies \mathbb{E}[S_3^2] = 30^2 + 15^2 = 1125\text{ min}^2 = 0.3125\text{ hr}^2$.
  - Utilization: $\rho_3 = 0.5 \times 0.5 = 0.25$.
  - Delay Cost: $C_3 = \$10/\text{hr}$ (batch nightly reporting).

Total Utilization:

$$
\rho = \rho_1 + \rho_2 + \rho_3 = 0.20 + 0.40 + 0.25 = 0.85 \quad (85.0\%)
$$

### Step 1: Compute Mean Residual Service Time $W_0$

$$
W_0 = \sum_{i=1}^3 \frac{\lambda_i \mathbb{E}[S_i^2]}{2}
$$

In minutes:
- Class 1: $\frac{12 \times 1.25}{2 \times 60} = \frac{15}{120} = 0.125\text{ minutes}$.
- Class 2: $\frac{3 \times 80.0}{2 \times 60} = \frac{240}{120} = 2.000\text{ minutes}$.
- Class 3: $\frac{0.5 \times 1125.0}{2 \times 60} = \frac{562.5}{120} = 4.6875\text{ minutes}$.

Summing all classes:

$$
W_0 = 0.125 + 2.000 + 4.6875 = 6.8125\text{ minutes}
$$

Notice that Class 3 (the elephants) contributes **$68.8\%$** of all residual waiting time!

---

### Step 2: Policy 1 — Standard First-In First-Out (FIFO)

Under FIFO, all jobs experience the identical expected queue waiting time regardless of class:

$$
\mathbb{E}[W_q^{\text{FIFO}}] = \frac{W_0}{1 - \rho} = \frac{6.8125\text{ min}}{1 - 0.85} = \frac{6.8125}{0.15} = 45.417\text{ minutes}
$$

Total Hourly Delay Cost under FIFO:

$$
\text{Cost}_{\text{FIFO}} = \sum_{i=1}^3 C_i \lambda_i \mathbb{E}[W_q]
$$

- Class 1 Cost: $\$300 \times 12 \times \frac{45.417}{60} = 3600 \times 0.7570 = \$2,725.00/\text{hr}$
- Class 2 Cost: $\$60 \times 3 \times \frac{45.417}{60} = 180 \times 0.7570 = \$136.25/\text{hr}$
- Class 3 Cost: $\$10 \times 0.5 \times \frac{45.417}{60} = 5 \times 0.7570 = \$3.79/\text{hr}$
- **Total FIFO Cost**: $\$2,725.00 + \$136.25 + \$3.79 = \mathbf{\$2,865.04 / \text{hour}}$

Notice that Class 1 (active engineers waiting on smoke tests) incurs **$95.1\%$ of the entire cost** because FIFO forces 1-minute smoke tests to wait 45 minutes behind 30-minute regression runs!

---

### Step 3: Policy 2 — The $c\mu$-Rule (Optimal Prioritization)

Let us calculate the $c\mu$ index for each class:
- **Class 1**: $C_1 \mu_1 = \$300 \times 60 = 18,000$
- **Class 2**: $C_2 \mu_2 = \$60 \times 7.5 = 450$
- **Class 3**: $C_3 \mu_3 = \$10 \times 2.0 = 20$

Ordering by descending $c\mu$: **Class 1 > Class 2 > Class 3**.
Cumulative utilizations:
- $\sigma_1 = \rho_1 = 0.20$
- $\sigma_2 = \rho_1 + \rho_2 = 0.20 + 0.40 = 0.60$
- $\sigma_3 = \rho_1 + \rho_2 + \rho_3 = 0.85$

Now compute expected queue wait times using Cobham's formula:

1. **Class 1 Waiting Time**:

$$
\mathbb{E}[W_{q, 1}] = \frac{W_0}{1 - \sigma_1} = \frac{6.8125\text{ min}}{1 - 0.20} = \frac{6.8125}{0.80} = \mathbf{8.516\text{ minutes}}
$$

   (Down from $45.42\text{ minutes}$ under FIFO — an **$81.3\%$ reduction** in developer waiting time!)

2. **Class 2 Waiting Time**:

$$
\mathbb{E}[W_{q, 2}] = \frac{W_0}{(1 - \sigma_1)(1 - \sigma_2)} = \frac{6.8125}{(1 - 0.20)(1 - 0.60)} = \frac{6.8125}{0.80 \times 0.40} = \frac{6.8125}{0.32} = \mathbf{21.289\text{ minutes}}
$$

   (Down from $45.42\text{ minutes}$ under FIFO — a **$53.1\%$ reduction**!)

3. **Class 3 Waiting Time**:

$$
\mathbb{E}[W_{q, 3}] = \frac{W_0}{(1 - \sigma_2)(1 - \sigma_3)} = \frac{6.8125}{(1 - 0.60)(1 - 0.85)} = \frac{6.8125}{0.40 \times 0.15} = \frac{6.8125}{0.06} = \mathbf{113.542\text{ minutes}}
$$

   (Up from $45.42\text{ minutes}$ to $1.89\text{ hours}$ — absorbing the heavy traffic).

---

### Step 4: Economic Cost Comparison

Hourly delay cost under $c\mu$-rule:
- Class 1 Cost: $\$300 \times 12 \times \frac{8.516}{60} = 3600 \times 0.14193 = \$510.96/\text{hr}$
- Class 2 Cost: $\$60 \times 3 \times \frac{21.289}{60} = 180 \times 0.35482 = \$63.87/\text{hr}$
- Class 3 Cost: $\$10 \times 0.5 \times \frac{113.542}{60} = 5 \times 1.89237 = \$9.46/\text{hr}$
- **Total $c\mu$ Cost**: $\$510.96 + \$63.87 + \$9.46 = \mathbf{\$584.29 / \text{hour}}$

### Final Financial Bottom Line:
- FIFO Hourly Cost: $\$2,865.04/\text{hr}$
- $c\mu$-Rule Hourly Cost: $\$584.29/\text{hr}$
- **Net Cost Savings**: **$\mathbf{\$2,280.75 / \text{hour}}$ ($79.6\%$ cost reduction)**!
- Annualized savings (2,000 engineering hours/year): **$\mathbf{\$4,561,500 / \text{year}}$**!
- All achieved with **zero hardware changes**, purely by ordering the priority queue according to the $c\mu$-rule.
